# Use Subqueries to Aggregate Data with the Warehouse Dataset

## Objective
- The objective of this query is to aggregate the data into a table containing each warehouse's ID, state and alias, and  number of orders; 
- as well as the grand total of orders for all warehouses combined; 
- and finally a column that classifies each warehouse by the percentage of grand total orders that it fulfilled: 0–20%, 21-60%, or > 60%. 

Note: This notebook breaks out the steps into manageable chunks. The final query is only intended to be run at the end. 

## Create Table
Here we use local csv files, so I use DuckDB for easier import. 
Let's name our table "orders" and "warehouse"

In [1]:
!pip install duckdb --trusted-host pypi.org --trusted-host files.pythonhosted.org

In [2]:
import duckdb

con = duckdb.connect("warehouse_orders.duckdb")
con.sql("""
CREATE TABLE orders AS
SELECT *
FROM read_csv_auto('WarehouseOrders_Orders.csv')
""")

In [3]:
con.sql("""
CREATE TABLE warehouse AS
SELECT *
FROM read_csv_auto('WarehouseOrders_Warehouse.csv')
""")

In [4]:
con.sql("""
SELECT *
FROM orders
LIMIT 10;
""")

┌──────────┬─────────────┬──────────────┬────────────┬──────────────┐
│ order_id │ customer_id │ warehouse_id │ order_date │ shipper_date │
│  int64   │    int64    │    int64     │    date    │     date     │
├──────────┼─────────────┼──────────────┼────────────┼──────────────┤
│      789 │        3731 │         8118 │ 2019-01-01 │ 2019-01-04   │
│      790 │        3486 │         8118 │ 2019-01-01 │ 2019-01-04   │
│      791 │        2623 │         8118 │ 2019-01-01 │ 2019-01-04   │
│      792 │        9869 │         8118 │ 2019-01-01 │ 2019-01-04   │
│      793 │        6866 │         8118 │ 2019-01-01 │ 2019-01-04   │
│      794 │        8055 │         8118 │ 2019-01-01 │ 2019-01-04   │
│      795 │        1152 │         8118 │ 2019-01-01 │ 2019-01-04   │
│      796 │        5765 │         8118 │ 2019-01-01 │ 2019-01-04   │
│      797 │        6709 │         8118 │ 2019-01-01 │ 2019-01-04   │
│      798 │        4866 │         2666 │ 2019-01-01 │ 2019-01-04   │
└──────────┴────────

In [5]:
con.sql("""
SELECT *
FROM warehouse
LIMIT 10;
""")

┌──────────────┬──────────────────────────────┬──────────────────┬────────────────┬─────────┐
│ warehouse_id │       warehouse_alias        │ maximum_capacity │ employee_total │  state  │
│    int64     │           varchar            │      int64       │     int64      │ varchar │
├──────────────┼──────────────────────────────┼──────────────────┼────────────────┼─────────┤
│         1543 │ Somerset Fulfillment Center  │              210 │             14 │ KY      │
│         2270 │ Bowling Green Warehouse      │              280 │             13 │ KY      │
│         2666 │ Lansing Fulfillment Center   │              290 │             16 │ MI      │
│         3417 │ Gatlinburg Warehouse         │              620 │              6 │ TN      │
│         3961 │ Lansing Storage Warehouse    │              740 │             22 │ MI      │
│         4338 │ Knoxville Fulfillment Center │              215 │             13 │ TN      │
│         6509 │ Memphis Fulfillment Center   │             

## Step 1: Combine and alias the tables
Aliasing is when we temporarily name a table or column in our query to make it easier to read and write. To alias the warehouse and orders tables and join the tables.

We will combine the two tables (warehouse and orders) using warehouse_id as the common key (the column shared by both tables).

## Step 2: Organize our new table
Use the GROUP BY clause in SQL to group rows that have the same values in specified columns into aggregated data, such as sum, count, average, maximum, or minimum, based on the values in another column. This operation is particularly useful in databases where there is a need to analyze data based on certain criteria. 

Here, we want the combined table to be grouped first by the warehouse ID and then by its name.

## Step 3: Build subquery logic
Now that we have the FROM statement and JOIN, we can go back up to the first lines and define the rows to select and operations to perform on them. \
From the objective, we know we want to return five columns: each warehouse's ID (warehouse_id—column 1), state and alias (this info will be combined into a single column: warehouse_name— column 2), and number of orders (number_of_orders—column 3); as well as the grand total of orders for all warehouses combined (total_orders—column 4); and finally a column that classifies each warehouse by the percentage of grand total orders that it fulfilled: 0–20%, 21-60%, or > 60% (fulfillment_summary—column 5). 

To create the final column, we'll need to use a special keyword.

## Step 4: Create categories using CASE
Use the CASE keyword in SQL to create categories or group data based on specific conditions. This is valuable when dealing with numerical or textual data that needs to be segmented into different groups or categories for analysis, reporting, or visualization purposes. 

For the final column, we'll use CASE to define which label to apply to each warehouse's fulfillment percentage (the percentage of the grand total of orders that it fulfilled). There will be three conditions, and thus three possible labels: "Fulfilled 0–20% of Orders", "Fulfilled 21–60% of Orders", or "Fulfilled more than 60% of Orders".

## Step 5: Filter using HAVING
Use the HAVING clause in SQL in combination with the GROUP BY clause to filter the results of aggregate functions in a query. While the WHERE clause filters individual rows before they are grouped, the HAVING clause filters groups of rows after they have been grouped. We filter out the warehouses that are currently being built (and therefore have no orders).

Here is the final query:

In [6]:
con.sql("""
SELECT
  Warehouse.warehouse_id,
  CONCAT(Warehouse.state, ': ', Warehouse.warehouse_alias) AS warehouse_name,
  COUNT(Orders.order_id) AS number_of_orders,
  (SELECT COUNT(*) FROM orders AS Orders) AS total_orders,
  CASE
    WHEN COUNT(Orders.order_id)/(SELECT COUNT(*) FROM orders AS Orders) <= 0.20
    THEN 'Fulfilled 0-20% of Orders'
    WHEN COUNT(Orders.order_id)/(SELECT COUNT(*) FROM orders AS Orders) > 0.20
    AND COUNT(Orders.order_id)/(SELECT COUNT(*) FROM orders AS Orders) <= 0.60
    THEN 'Fulfilled 21-60% of Orders'
    ELSE 'Fulfilled more than 60% of Orders'
  END AS fulfillment_summary
FROM warehouse AS Warehouse
LEFT JOIN orders AS Orders
ON Orders.warehouse_id = Warehouse.warehouse_id
GROUP BY
  Warehouse.warehouse_id,
  warehouse_name
HAVING
  COUNT(Orders.order_id) > 0
""")

┌──────────────┬──────────────────────────────────┬──────────────────┬──────────────┬────────────────────────────┐
│ warehouse_id │          warehouse_name          │ number_of_orders │ total_orders │    fulfillment_summary     │
│    int64     │             varchar              │      int64       │    int64     │          varchar           │
├──────────────┼──────────────────────────────────┼──────────────────┼──────────────┼────────────────────────────┤
│         9080 │ KY: Frankfort Fulfillment Center │              500 │         9999 │ Fulfilled 0-20% of Orders  │
│         4338 │ TN: Knoxville Fulfillment Center │              343 │         9999 │ Fulfilled 0-20% of Orders  │
│         1543 │ KY: Somerset Fulfillment Center  │              548 │         9999 │ Fulfilled 0-20% of Orders  │
│         2666 │ MI: Lansing Fulfillment Center   │             3178 │         9999 │ Fulfilled 21-60% of Orders │
│         6509 │ TN: Memphis Fulfillment Center   │             2403 │         9